In [1]:
import pandas as pd
import pickle
import os

In [20]:
base = '/data/bwh-comppath-img2/MGH_CID/public_dataset/physionet.org/files/ehrcon-consistency-of-notes/1.0.1'
os.chdir(base)

In [30]:
hadm_id = '138312.0'
with open(f'original/test/discharge/discharge_label/EHRCon_{hadm_id}_data.pkl', 'rb') as f:
    data = pickle.load(f)

# df = pd.read_csv("/data/bwh-comppath-img2/MGH_CID/public_dataset/physionet.org/files/ehrcon-consistency-of-notes/1.0.1/original/valid/discharge/discharge_val.csv")
# df['HADM_ID'] = df['HADM_ID'].astype(str)
# df[df['HADM_ID'] == hadm_id].TEXT.values[0]

In [31]:
data

{11669: [{'Temperature': {'data': [{'chartevents': {'valuenum': '98.6',
       'charttime': 'n'},
      'd_items': {'label': 'temperature f'},
      'label': "'valuenum' and 'charttime' are inconsistency",
      'errors': 2}],
    'position': 74,
    'source': 'mimic',
    'entity_type': '1',
    'all_queries': [{'Temperature': {'chartevents': {'valuenum': '98.6',
        'charttime': 'n'},
       'd_items': {'label': 'temperature f'}},
      'label': "'valuenum' and 'charttime' are inconsistency",
      'errors': 2},
     {'Temperature': {'chartevents': {'valuenum': '98.6', 'charttime': 'n'},
       'd_items': {'label': 'temperature f (calc)'}},
      'label': "'valuenum' and 'label' and 'charttime' are inconsistency",
      'errors': 3},
     {'Temperature': {'chartevents': {'valuenum': '98.6', 'charttime': 'n'},
       'd_items': {'label': 'temperature fahrenheit'}},
      'label': "'valuenum' and 'label' and 'charttime' are inconsistency",
      'errors': 3}]}},
  {'blood pressure'

In [2]:
import pickle
import pandas as pd

KEEP_FIELDS = [
    "charttime", "starttime", "endtime", "startdate", "enddate",
    "valuenum", "valueuom",
    "dose_val_rx", "dose_unit_rx",
    "amount", "amountuom",
    "rate", "rateuom",
    "drug", "route", "originalroute", "form_unit_disp",
    "org_name", "spec_type_desc"
]

IGNORE_KEYS = {
    "label", "errors",
    "d_items", "d_labitems", "d_icd_diagnoses", "d_icd_procedures"
}

def clean(x):
    if x is None:
        return None
    if isinstance(x, str) and x.strip().lower() in {"nan", "none", ""}:
        return None
    return x

def flatten_note_only(pkl_file, out_csv=None):
    with open(pkl_file, "rb") as f:
        obj = pickle.load(f)

    rows = []

    for note_id, entity_list in obj.items():
        for item in entity_list:
            if not isinstance(item, dict) or len(item) != 1:
                continue

            entity_text, meta = next(iter(item.items()))
            position = meta.get("position")
            entity_type = meta.get("entity_type")

            data_entries = meta.get("data", [])

            # If no extracted fields, still keep the entity mention
            if not data_entries:
                row = {
                    "note_id": note_id,
                    "position": position,
                    "entity_text": entity_text,
                    "entity_type": entity_type,
                }
                for f in KEEP_FIELDS:
                    row[f] = None
                rows.append(row)
                continue

            for entry in data_entries:
                row = {
                    "note_id": note_id,
                    "position": position,
                    "entity_text": entity_text,
                    "entity_type": entity_type,
                }
                for f in KEEP_FIELDS:
                    row[f] = None

                # collect only raw note-extracted payload fields
                for k, v in entry.items():
                    if k in IGNORE_KEYS:
                        continue
                    if isinstance(v, dict):
                        for subk, subv in v.items():
                            if subk in KEEP_FIELDS:
                                row[subk] = clean(subv)

                rows.append(row)

    df = pd.DataFrame(rows)

    # optional de-dup
    dedup_cols = ["note_id", "position", "entity_text", "entity_type"] + KEEP_FIELDS
    df = df.drop_duplicates(subset=dedup_cols)

    if out_csv:
        df.to_csv(out_csv, index=False)

    return df

In [17]:
hadm_id = '117833.0'

df_o = flatten_note_only(f'original/test/physician/physician_label/EHRCon_{hadm_id}_data.pkl')
print(len(df_o))
df_p = flatten_note_only(f'processed/test/physician/physician_label/EHRCon_{hadm_id}_data.pkl')
print(len(df_p))

62
62


In [ ]:
df_p

In [13]:

df_p = flatten_note_only(f'processed/test/nursing/nursing_label/EHRCon_{hadm_id}_data.pkl')
df_p

,note_id,position,entity_text,entity_type,charttime,starttime,endtime,startdate,enddate,valuenum,...,amount,amountuom,rate,rateuom,drug,route,originalroute,form_unit_disp,org_name,spec_type_desc
0,720692,24,Height,1,chart,None,None,None,None,65,...,None,None,None,None,None,None,None,None,None,None
1,720692,26,Admission weight,1,adm,None,None,None,None,92.7,...,None,None,None,None,None,None,None,None,None,None
2,720692,28,Daily weight,1,chart,None,None,None,None,97.4,...,None,None,None,None,None,None,None,None,None,None
3,720692,36,Non-invasive BP,1,chart,None,None,None,None,121,...,None,None,None,None,None,None,None,None,None,None
4,720692,36,Non-invasive BP,1,chart,None,None,None,None,55,...,None,None,None,None,None,None,None,None,None,None
5,720692,39,Temperature,1,chart,None,None,None,None,97.6,...,None,None,None,None,None,None,None,None,None,None
6,720692,41,Arterial BP,1,chart,None,None,None,None,109,...,None,None,None,None,None,None,None,None,None,None
7,720692,41,Arterial BP,1,chart,None,None,None,None,103,...,None,None,None,None,None,None,None,None,None,None
8,720692,44,Respiratory rate,1,chart,None,None,None,None,21,...,None,None,None,None,None,None,None,None,None,None
9,720692,46,Heart Rate,1,chart,None,None,None,None,76,...,None,None,None,None,None,None,None,None,None,None


In [6]:

df = pd.read_csv("nursing_test.csv")
df['HADM_ID'] = df['HADM_ID'].astype(str)
df[df['HADM_ID'] == hadm_id].TEXT.values[0]

'59 yo woman with mental retardation with h/o afib with RVR, chronic\n   ileus, mediastinal mass determined Hodgkin\ns. Cardiac tamponade 1 month\n   ago s/p pericardiocentesis.  Admitted [**7-16**] for tachypnia & hypoxemia.\n   Treated for pna empirically. Became hypoxemic and tachypnic [**7-20**] with\n   subsequent [**Hospital 852**] transferred to MICU. Pt has bilateral pleural effusions\n   as well as pericardial effusions. Afib tx\nd with dilt and esmolol gtts,\n   converted and to NSR with dilt gtt, currently receiving PO metoprolol\n   for rate control. Pt received first dose of chemotherapy on [**7-21**] on\n   modified Hodgkin\ns lymphoma therapy not receiving vincristine or\n   bleomycin. Last received chemo (etoposide) [**7-23**]. Chemo precautions\n   d/c\nd. Intubated [**Date range (1) 3218**] for resp distress secondary to large pleural\n   effusion and hypercarbia. Effusion tapped with good effect 1.1L\n   removed. Extubated [**7-26**] maintaining sats on NC and transf